In [1]:
# Persiapan lingkungan; sel pendamping, bukan listing buku.
import sys
import numpy as np

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('Dependensi berhasil dimuat. Tidak ada dataset eksternal yang diperlukan.')

Python: 3.13.16
NumPy: 2.1.3
Dependensi berhasil dimuat. Tidak ada dataset eksternal yang diperlukan.


## 2. Ringkasan Chapter 3

Chapter 3 menjelaskan tahap predict dalam alur predict, compare, dan learn. Prediksi pertama dibuat dengan mengalikan satu masukan dengan satu bobot. Bobot menentukan bagaimana informasi masukan memengaruhi keluaran; pada contoh paling sederhana, bobot berfungsi sebagai faktor pengali.

Pembahasan berkembang ke beberapa input untuk satu output. Setiap input dikalikan dengan bobot yang sesuai, lalu semua kontribusi dijumlahkan. Operasi ini disebut weighted sum atau dot product. Buku memperkenalkan vektor sebagai representasi kumpulan angka dan memberikan tantangan implementasi operasi elementwise, penjumlahan vektor, serta rata-rata vektor.

Untuk satu input dan beberapa output, input dikalikan dengan masing-masing bobot sehingga menghasilkan vektor prediksi. Untuk beberapa input dan output, setiap output memperoleh weighted sum tersendiri; bobotnya disimpan dalam matriks. Keluaran satu lapisan selanjutnya dapat menjadi masukan lapisan lain, sehingga terbentuk jaringan berlapis.

NumPy digunakan untuk melakukan operasi yang sama dengan penulisan yang lebih ringkas. Memahami shape, transpose, broadcasting, dan perbedaan perkalian elementwise dengan dot product menjadi penting agar operasi numerik sesuai dengan tujuan model. Seluruh contoh menggunakan bobot yang ditetapkan sebelumnya. Bab ini belum melatih bobot atau membuktikan akurasi pada suatu dataset nyata.

**Rujukan:** Trask (2019), hlm. 21-46.

## 3. Peta subbab dan reproduksi

| Subbab dalam buku | Halaman cetak | Pengerjaan dalam notebook |
| --- | --- | --- |
| Step 1: Predict | 22-23 | Input, output, bobot, dan bentuk data |
| A simple neural network making a prediction | 24 | Prediksi satu input dan satu output |
| What is a neural network? | 25 | Penjelasan fungsi dan keterbatasan contoh |
| What does this neural network do? | 26-27 | Skala, sensitivitas, dan bobot negatif |
| Making a prediction with multiple inputs | 28-29 | Weighted sum dengan Python |
| Multiple inputs: What does this neural network do? | 30-34 | Vektor, kontribusi input, dot product, dan tantangan vektor |
| Multiple inputs: Complete runnable code | 35 | Versi Python dan NumPy |
| Making a prediction with multiple outputs | 36-37 | Perkalian satu input dengan beberapa bobot |
| Predicting with multiple inputs and outputs | 38-39 | Tiga weighted sum dengan matriks bobot |
| Multiple inputs and outputs: How does it work? | 40-41 | Orientasi matriks dan helper perkalian |
| Predicting on predictions | 42-43 | Jaringan berlapis versi list dan NumPy |
| A quick primer on NumPy | 44-46 | Array, broadcasting, shape, dan contoh error |
| Summary | 46 | Forward propagation dan kesiapan menuju Chapter 4 |

Listing yang berulang pada diagram buku digabung menjadi sel kode lengkap. Nilai input dan bobot inti dipertahankan. Sel yang diberi label tambahan memperjelas teori atau memeriksa hasil, bukan diklaim sebagai listing asli.

## 4. Step 1: Predict

Untuk membuat prediksi, model memerlukan informasi masukan, bobot, dan aturan pengolahan. Pada bentuk sederhana:

$$
\hat{y}=xw
$$

$x$ adalah input, $w$ adalah bobot, dan $\hat{y}$ adalah hasil prediksi. Rumus tersebut merupakan notasi pendamping untuk menjelaskan operasi perkalian yang ditulis dalam kode buku.

**Feature berbeda dari sample.** Pada contoh beberapa input, tiga angka seperti jumlah jari kaki, persentase kemenangan, dan jumlah penggemar adalah tiga feature untuk **satu pertandingan**. Empat nilai pada setiap list menyatakan empat observasi pertandingan. Karena itu, banyak input pada satu prediksi tidak otomatis berarti banyak pertandingan diproses sekaligus.

Pilihan bentuk jaringan bergantung pada jumlah feature dan jumlah keluaran yang diperlukan. Pada tahap ini, bobot sudah diberikan dan tidak berubah selama prediksi.

**Rujukan:** Trask (2019), hlm. 22-23 dan 28-31.

## 5. A simple neural network making a prediction

**Asal kode:** reproduksi gabungan listing hlm. 24-25. Nama `input` diubah menjadi `input_data`; logika dan angka inti dipertahankan.

Model menggunakan bobot `0.1` dan input pertama dari `number_of_toes`, yaitu `8.5`. Nilai matematis prediksinya adalah $8.5\times0.1=0.85$. Fungsi mengembalikan hasil perkalian dan `print` menampilkannya.

In [2]:
# Reproduksi: satu input dan satu output, hlm. 24-25.
weight = 0.1

def neural_network(input_data, weight):
    prediction = input_data * weight
    return prediction

number_of_toes = [8.5, 9.5, 10, 9]
input_data = number_of_toes[0]
pred = neural_network(input_data, weight)
print(pred)

single_pred = pred
single_input = input_data
single_model = neural_network

0.8500000000000001


### Penjelasan teori dan hasil

`number_of_toes[0]` mengambil observasi pertama, bukan seluruh list. Parameter `weight` menentukan faktor pengali yang diterapkan pada angka tersebut. Belum ada target pertandingan, fungsi loss, atau pembaruan bobot.

**Floating point:** Python dapat menampilkan `0.8500000000000001`. Ini merupakan perbedaan representasi numerik dari nilai matematis 0.85, bukan bukti bahwa formula salah. Untuk pemeriksaan hasil, notebook menggunakan toleransi numerik pada bagian verifikasi.

**What is a neural network?** Definisi sementara buku berupa bobot yang mengubah input menjadi prediksi merupakan titik masuk pembelajaran. Contoh satu perkalian ini belum mencakup keseluruhan komponen jaringan saraf yang lebih kompleks, seperti bias, aktivasi nonlinear, dan prosedur belajar.

Input contoh tidak membuktikan bahwa rata-rata jumlah jari kaki benar-benar merupakan prediktor kemenangan yang baik. Buku menggunakannya untuk menjelaskan mekanisme perhitungan.

**Rujukan:** Trask (2019), hlm. 24-25. Klarifikasi numerik dan batas definisi merupakan pendalaman tambahan.

## 6. What does this neural network do?

Buku mengibaratkan bobot sebagai pengatur volume. Untuk model $\hat{y}=xw$, bobot memperbesar atau memperkecil input dan dapat mengubah tandanya.

| Bobot | Efek pada input positif |
| --- | --- |
| 2 | Menggandakan nilai |
| 0.01 | Menghasilkan seperseratus nilai |
| 0 | Menghasilkan nol |
| Negatif | Menghasilkan nilai dengan tanda berlawanan |

**Pendalaman sensitivitas:** Pada fungsi sederhana ini, perubahan input sebesar $\Delta x$ mengubah output sebesar $w\Delta x$. Besar dan tanda bobot sama-sama berperan. Jika input dan bobot keduanya negatif, hasil perkalian positif.

Fungsi contoh bersifat stateless: pemanggilan baru menggunakan input dan bobot yang diberikan saat itu. Pernyataan ini berlaku untuk kode di sini, bukan semua arsitektur neural network yang mungkin memiliki state atau memori.

**Asal kode berikut:** eksplorasi tambahan berdasarkan penjelasan hlm. 26-27.

In [3]:
# Tambahan: mengamati efek bobot, tanpa pelatihan.
for trial_weight in [2.0, 0.01, 0.0, -0.1]:
    value = single_model(8.5, trial_weight)
    print(f'input=8.5, weight={trial_weight}: prediction={value:.4f}')

# Ilustrasi input negatif pada hlm. 27: hasilnya sebuah skor numerik.
print('input=-10, weight=-8.9:', single_model(-10, -8.9))

input=8.5, weight=2.0: prediction=17.0000
input=8.5, weight=0.01: prediction=0.0850
input=8.5, weight=0.0: prediction=0.0000
input=8.5, weight=-0.1: prediction=-0.8500
input=-10, weight=-8.9: 89.0


## 7. Making a prediction with multiple inputs

Untuk beberapa input dan satu output, model mengalikan setiap feature dengan bobot pada posisi yang sama, lalu menjumlahkan hasilnya:

$$
\hat{y}=\sum_{i=1}^{d} x_i w_i
$$

Perhitungan untuk pertandingan pertama adalah:

| Feature | Input | Bobot | Kontribusi |
| --- | --- | --- | --- |
| toes | 8.5 | 0.1 | 0.85 |
| wlrec | 0.65 | 0.2 | 0.13 |
| nfans | 1.2 | 0 | 0 |
| Total | | | 0.98 |

`wlrec` menggunakan proporsi, sehingga 65% ditulis sebagai `0.65`. `nfans` menggunakan satuan juta penggemar. Urutan feature harus sama dengan urutan bobot.

**Asal kode:** reproduksi lengkap hlm. 28-30 dan 35, menggunakan Python list. Fungsi `w_sum` memeriksa kesamaan panjang vektor, menghitung setiap perkalian, dan mengakumulasikan hasilnya.

In [4]:
# Reproduksi: banyak input, satu output, hlm. 28-30 dan 35.
def w_sum(a, b):
    assert len(a) == len(b)
    output = 0
    for i in range(len(a)):
        output += a[i] * b[i]
    return output

weights = [0.1, 0.2, 0]

def neural_network(input_data, weights):
    pred = w_sum(input_data, weights)
    return pred

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]
input_data = [toes[0], wlrec[0], nfans[0]]
pred = neural_network(input_data, weights)
print(pred)

multi_input_pred_python = pred
multi_input_values = input_data.copy()
multi_input_weights = weights.copy()
multi_input_model = neural_network

0.9800000000000001


### Multiple inputs: What does this neural network do?

Weighted sum dan dot product merujuk pada operasi yang sama untuk vektor real di contoh ini. Model tidak hanya memeriksa besar bobot, tetapi juga hasil perkalian input dan bobot.

Walaupun bobot `wlrec` lebih besar daripada bobot `toes`, kontribusi `toes` pada observasi ini lebih besar karena nilai inputnya lebih besar. Bobot `nfans` bernilai nol, sehingga feature tersebut tidak memengaruhi hasil contoh ini.

Bobot tidak dapat diacak tanpa sekaligus mengubah pasangan feature yang sesuai. Sebagai contoh, bobot untuk proporsi kemenangan tidak boleh diterapkan ke jumlah penggemar karena posisi datanya tertukar.

**Pendalaman:** Nilai bobot mentah pada feature dengan satuan berbeda tidak langsung menunjukkan kepentingan yang dapat dibandingkan secara adil. Selain itu, keluaran weighted sum tidak otomatis berada pada rentang 0-1 atau menjadi probabilitas yang telah terkalibrasi.

**Rujukan:** Trask (2019), hlm. 30-34; catatan rentang keluaran merupakan pendalaman tambahan.

## 8. Challenge: Vector math

Buku meminta pembaca menulis empat fungsi: `elementwise_multiplication`, `elementwise_addition`, `vector_sum`, dan `vector_average`, lalu menggunakan dua di antaranya untuk menghitung dot product.

**Solusi tantangan pendamping:** buku memberikan nama dan tugas fungsi, bukan listing solusi ini. Implementasi berikut dibuat untuk memenuhi tantangan tersebut.

Perkalian elementwise mengembalikan **vektor** hasil pasangan perkalian. Dot product menjumlahkan elemen vektor tersebut sehingga menghasilkan **satu skalar**. Kedua operasi tidak boleh disamakan.

**Rujukan:** Trask (2019), hlm. 31.

In [5]:
# Solusi tantangan hlm. 31: implementasi pendamping.
def elementwise_multiplication(vec_a, vec_b):
    if len(vec_a) != len(vec_b):
        raise ValueError('Kedua vektor harus memiliki panjang yang sama.')
    return [vec_a[i] * vec_b[i] for i in range(len(vec_a))]

def elementwise_addition(vec_a, vec_b):
    if len(vec_a) != len(vec_b):
        raise ValueError('Kedua vektor harus memiliki panjang yang sama.')
    return [vec_a[i] + vec_b[i] for i in range(len(vec_a))]

def vector_sum(vec_a):
    total = 0
    for value in vec_a:
        total += value
    return total

def vector_average(vec_a):
    if len(vec_a) == 0:
        raise ValueError('Rata-rata tidak dapat dihitung untuk vektor kosong.')
    return vector_sum(vec_a) / len(vec_a)

def dot_from_vector_operations(vec_a, vec_b):
    return vector_sum(elementwise_multiplication(vec_a, vec_b))

vec_a = [1, 2, 3]
vec_b = [4, 5, 6]
print('Elementwise multiplication:', elementwise_multiplication(vec_a, vec_b))
print('Elementwise addition:', elementwise_addition(vec_a, vec_b))
print('Vector sum:', vector_sum(vec_a))
print('Vector average:', vector_average(vec_a))
print('Dot product:', dot_from_vector_operations(vec_a, vec_b))

Elementwise multiplication: [4, 10, 18]
Elementwise addition: [5, 7, 9]
Vector sum: 6
Vector average: 2.0
Dot product: 32


## 9. Intuisi dot product dan batas analogi logika

Buku memberikan beberapa vektor untuk menunjukkan tumpang tindih kontribusi dan pembatalan oleh tanda negatif. Listing berikut menjalankan perhitungan yang dituliskan pada hlm. 32, ditambah contoh yang dibahas pada hlm. 33.

Pada hlm. 32, `e` adalah `[0, 1, -1, 0]`; pada hlm. 33, buku menggunakan contoh `e` berbeda, yaitu `[-1, 1, 0, 0]`. Notebook menamai contoh kedua `e_logic` agar kedua ilustrasi tidak tertukar.

In [6]:
# Reproduksi contoh dot product, hlm. 32-33.
a = [0, 1, 0, 1]
b = [1, 0, 1, 0]
c = [0, 1, 1, 0]
d = [0.5, 0, 0.5, 0]
e = [0, 1, -1, 0]

dot_examples = {
    'a dot b': w_sum(a, b),
    'b dot c': w_sum(b, c),
    'b dot d': w_sum(b, d),
    'c dot c': w_sum(c, c),
    'd dot d': w_sum(d, d),
    'c dot e': w_sum(c, e),
    'e dot e': w_sum(e, e),
    'c dot d': w_sum(c, d),
}
for label, value in dot_examples.items():
    print(f'{label}: {value}')

e_logic = [-1, 1, 0, 0]
print('d dot e_logic:', w_sum(d, e_logic))

a dot b: 0
b dot c: 1
b dot d: 1.0
c dot c: 2
d dot d: 0.5
c dot e: 0
e dot e: 2
c dot d: 0.5
d dot e_logic: -0.5


### Penjelasan teori

- `a dot b = 0` karena tidak ada posisi yang menghasilkan kontribusi bukan nol.
- `c dot e = 0` terjadi melalui kontribusi positif dan negatif yang saling membatalkan. Jadi, hasil nol tidak selalu berarti tidak ada elemen yang berkontribusi.
- `e dot e = 2` memperlihatkan bahwa perkalian dua angka negatif menghasilkan kontribusi positif.

Buku menggunakan AND, OR, dan NOT sebagai analogi membaca kontribusi bobot. Weighted sum tetap merupakan operasi numerik; bobot negatif tidak otomatis menjadi gerbang Boolean NOT, dan penjumlahan tidak otomatis merupakan OR Boolean.

**Pendalaman similarity:** Dot product dipengaruhi oleh arah dan besar vektor. Vektor identik tidak selalu memberikan skor tertinggi jika vektor pembanding memiliki skala lebih besar. Contoh tambahan berikut memperlihatkan batas pernyataan tersebut tanpa mengubah reproduksi buku.

**Rujukan:** Trask (2019), hlm. 32-34. Klarifikasi similarity merupakan pendalaman matematis tambahan.

In [7]:
# Tambahan: dot product dipengaruhi besar vektor.
u = [1, 1]
v = [100, 100]
print('u dot u:', w_sum(u, u))
print('u dot v:', w_sum(u, v))
print('Skor yang lebih besar tidak sendirian membuktikan kemiripan yang lebih baik.')

u dot u: 2
u dot v: 200
Skor yang lebih besar tidak sendirian membuktikan kemiripan yang lebih baik.


## 10. Multiple inputs: Complete runnable code - NumPy

**Asal kode:** reproduksi versi NumPy pada hlm. 35. Fungsi `input_data.dot(weights)` melakukan operasi weighted sum yang sebelumnya dibuat menggunakan loop Python.

Untuk dua array 1D, dot product menghasilkan skalar. `input_data * weights` justru menghasilkan array perkalian elementwise; hasilnya baru setara dengan dot product jika kemudian dijumlahkan.

**Rujukan:** Trask (2019), hlm. 35; [dokumentasi NumPy `dot`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html).

In [8]:
# Reproduksi versi NumPy, hlm. 35.
import numpy as np

weights = np.array([0.1, 0.2, 0])

def neural_network(input_data, weights):
    pred = input_data.dot(weights)
    return pred

toes = np.array([8.5, 9.5, 9.9, 9.0])
wlrec = np.array([0.65, 0.8, 0.8, 0.9])
nfans = np.array([1.2, 1.3, 0.5, 1.0])
input_data = np.array([toes[0], wlrec[0], nfans[0]])
pred = neural_network(input_data, weights)
print(pred)

multi_input_pred_numpy = pred
print('Elementwise contributions:', input_data * weights)
print('Sum of contributions:', (input_data * weights).sum())

0.9800000000000001
Elementwise contributions: [0.85 0.13 0.  ]
Sum of contributions: 0.9800000000000001


## 11. Making a prediction with multiple outputs

Untuk satu input dan beberapa output, masing-masing keluaran memperoleh perkalian sendiri:

$$
\hat{y}_j=xw_j
$$

Pada contoh buku, input `0.65` dikalikan dengan bobot `[0.3, 0.2, 0.9]`, menghasilkan nilai teoritis `[0.195, 0.13, 0.585]` untuk urutan hurt, win, dan sad. Tidak ada penjumlahan ketiga angka tersebut karena ketiganya merupakan keluaran terpisah.

**Asal kode:** reproduksi gabungan hlm. 36-37. Salah ketik pemanggilan `weight` pada hlm. 37 diperbaiki menjadi `weights`. Helper `ele_mul` tetap mengikuti contoh tiga output pada buku.

In [9]:
# Reproduksi: satu input, tiga output, hlm. 36-37.
def ele_mul(number, vector):
    output = [0, 0, 0]
    assert len(output) == len(vector)
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

weights = [0.3, 0.2, 0.9]

def neural_network(input_data, weights):
    pred = ele_mul(input_data, weights)
    return pred

wlrec = [0.65, 0.8, 0.8, 0.9]
input_data = wlrec[0]
pred = neural_network(input_data, weights)
print(pred)

multi_output_pred_python = pred.copy()
print('Output order: hurt, win, sad')

[0.195, 0.13, 0.5850000000000001]
Output order: hurt, win, sad


### Penjelasan hasil

Ketiga output memakai input yang sama, tetapi memiliki bobot berbeda. Pemisahan komponen perhitungan ini tidak merupakan bukti bahwa kejadian hurt, win, dan sad independen secara statistik. Label tersebut hanya memberi konteks bagi contoh.

Tidak ada syarat bahwa ketiga nilai harus berjumlah satu, karena labelnya bukan tiga kategori yang harus saling eksklusif. Bahkan berada pada rentang 0-1 pun belum membuktikan angka tersebut merupakan probabilitas yang terkalibrasi.

Helper `ele_mul` memakai list output dengan panjang tiga. Ini sesuai reproduksi contoh buku, tetapi bukan implementasi umum untuk semua jumlah output.

**Rujukan:** Trask (2019), hlm. 36-37. Batas interpretasi output merupakan pendalaman tambahan.

## 12. Predicting with multiple inputs and outputs

Setiap output menghitung dot product antara input dan satu baris bobot:

$$
\hat{y}_j=\sum_{i=1}^{d} W_{ji}x_i
$$

Pada representasi list buku, **baris matriks adalah output** dan **kolom adalah input**.

| Output | Bobot toes | Bobot wlrec | Bobot nfans | Nilai teoritis |
| --- | --- | --- | --- | --- |
| hurt | 0.1 | 0.1 | -0.3 | 0.555 |
| win | 0.1 | 0.2 | 0 | 0.98 |
| sad | 0 | 1.3 | 0.1 | 0.965 |

**Asal kode:** reproduksi hlm. 38-41. Pemanggilan `weight` pada salah satu potongan hlm. 39 diperbaiki menjadi `weights`. Helper asli yang mengasumsikan contoh persegi 3×3 dipertahankan di sini; versi umum dibahas terpisah.

In [10]:
# Reproduksi: tiga input, tiga output, hlm. 38-41.
def vect_mat_mul(vect, matrix):
    assert len(vect) == len(matrix)
    output = [0, 0, 0]
    for i in range(len(vect)):
        output[i] = w_sum(vect, matrix[i])
    return output

weights = [
    [0.1, 0.1, -0.3],  # hurt
    [0.1, 0.2, 0.0],   # win
    [0.0, 1.3, 0.1],   # sad
]

def neural_network(input_data, weights):
    pred = vect_mat_mul(input_data, weights)
    return pred

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]
input_data = [toes[0], wlrec[0], nfans[0]]
pred = neural_network(input_data, weights)
print(pred)

multi_io_pred_python = pred.copy()
multi_io_weights_rows = [row.copy() for row in weights]
multi_io_input = input_data.copy()

[0.555, 0.9800000000000001, 0.9650000000000001]


### Multiple inputs and outputs: How does it work?

Setiap baris matriks menghasilkan satu prediksi. Perhitungan pertama, misalnya, adalah $8.5(0.1)+0.65(0.1)+1.2(-0.3)=0.555$.

Secara notasi dengan vektor kolom, representasi baris tersebut dapat ditulis sebagai $y=W_{\mathrm{rows}}x$. Pada implementasi NumPy buku, input digunakan di sebelah kiri dan bobot ditranspose, sehingga operasi yang setara adalah `x.dot(W_rows.T)`.

Kedua cara konsisten jika bentuk dan orientasinya dipertahankan. Mengubah urutan atau transpose tanpa memperhatikan definisi baris/kolom dapat menghasilkan nilai yang salah meskipun shape masih cocok, terutama pada matriks persegi.

**Rujukan:** Trask (2019), hlm. 40-41 dan catatan orientasi matriks pada hlm. 41.

## 13. Pendalaman: helper untuk matriks tidak persegi

Helper `vect_mat_mul` dari buku berjalan untuk contoh tiga input dan tiga output. Namun, jumlah baris matriks menyatakan jumlah output, sedangkan panjang input menyatakan jumlah feature. Keduanya tidak harus sama.

**Implementasi tambahan:** helper berikut memeriksa panjang setiap baris bobot dan mengembalikan satu hasil untuk setiap baris. Contoh menggunakan tiga input dan dua output, kemudian dibandingkan dengan NumPy. Ini merupakan pengembangan untuk menjelaskan batas helper buku, bukan pengganti diam-diam atas reproduksi asli.

In [11]:
# Tambahan: matriks berbentuk (n_outputs, n_inputs).
def vect_mat_mul_general(vect, matrix):
    output = []
    for row in matrix:
        if len(row) != len(vect):
            raise ValueError('Setiap baris bobot harus cocok dengan panjang input.')
        output.append(w_sum(vect, row))
    return output

rectangular_weights = [
    [0.1, 0.1, -0.3],
    [0.1, 0.2, 0.0],
]
rectangular_pred = vect_mat_mul_general(multi_io_input, rectangular_weights)
rectangular_numpy = np.array(multi_io_input).dot(np.array(rectangular_weights).T)
print('Three inputs, two outputs (list):', rectangular_pred)
print('Three inputs, two outputs (NumPy):', rectangular_numpy)

try:
    vect_mat_mul(multi_io_input, rectangular_weights)
except AssertionError:
    print('Helper asli menolak bentuk ini; batas contoh persegi berhasil diperlihatkan.')
else:
    raise AssertionError('Contoh seharusnya memperlihatkan batas helper asli.')

Three inputs, two outputs (list): [0.555, 0.9800000000000001]
Three inputs, two outputs (NumPy): [0.555 0.98 ]
Helper asli menolak bentuk ini; batas contoh persegi berhasil diperlihatkan.


## 14. Predicting on predictions - Python list

Buku menyusun dua pengolahan berurutan: input menghasilkan hidden layer, lalu hidden layer menghasilkan output. Nilai hidden layer adalah keluaran antara, bukan bobot yang baru dilatih.

Dengan orientasi baris seperti pada helper list:

$$
h=W_{ih}x,\qquad \hat{y}=W_{hp}h
$$

**Asal kode:** reproduksi gabungan hlm. 42-43. Input serta kedua matriks bobot dipertahankan. `print` hidden layer ditambahkan agar proses antara terlihat.

In [12]:
# Reproduksi jaringan berlapis versi list, hlm. 42-43.
ih_wgt = [
    [0.1, 0.2, -0.1],
    [-0.1, 0.1, 0.9],
    [0.1, 0.4, 0.1],
]
hp_wgt = [
    [0.3, 1.1, -0.3],
    [0.1, 0.2, 0.0],
    [0.0, 1.3, 0.1],
]
weights = [ih_wgt, hp_wgt]

def neural_network(input_data, weights):
    hid = vect_mat_mul(input_data, weights[0])
    pred = vect_mat_mul(hid, weights[1])
    return pred

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]
input_data = [toes[0], wlrec[0], nfans[0]]
hid = vect_mat_mul(input_data, weights[0])
pred = neural_network(input_data, weights)
print('Hidden layer:', hid)
print('Prediction:', pred)

stack_hidden_python = hid.copy()
stack_pred_python = pred.copy()
stack_weights_python = weights
stack_input = input_data.copy()

Hidden layer: [0.8600000000000001, 0.29499999999999993, 1.23]
Prediction: [0.21350000000000002, 0.14500000000000002, 0.5065]


### Penjelasan hidden layer dan hasil

Nilai teoritis hidden layer adalah `[0.86, 0.295, 1.23]`. Contoh perhitungan hidden pertama adalah $8.5(0.1)+0.65(0.2)+1.2(-0.1)=0.86$.

Output teoritisnya adalah `[0.2135, 0.145, 0.5065]`. Diagram pada hlm. 43 menampilkan sebagian nilai dengan pembulatan menjadi sekitar `0.214` dan `0.507`; output kode tidak harus persis sama dengan angka diagram yang telah dibulatkan.

**Activation dalam konteks bab ini:** angka yang diteruskan selama forward propagation, termasuk nilai hidden dan output. Istilah tersebut perlu dibedakan dari activation function nonlinear, yang belum digunakan pada contoh ini.

**Rujukan:** Trask (2019), hlm. 42-43 dan 46.

## 15. Predicting on predictions - NumPy

**Asal kode:** reproduksi versi NumPy hlm. 43. `.T` pada kedua matriks dipertahankan karena diperlukan untuk mengganti perkalian berbasis baris pada helper list dengan perkalian input di sebelah kiri.

| Variabel | Shape pada contoh | Makna |
| --- | --- | --- |
| input_data | `(3,)` | Tiga feature satu pertandingan |
| ih_wgt | `(3, 3)` | Bobot input menuju hidden, setelah transpose |
| hid | `(3,)` | Tiga nilai hidden |
| hp_wgt | `(3, 3)` | Bobot hidden menuju output, setelah transpose |
| pred | `(3,)` | Tiga keluaran |

Transpose menentukan orientasi nilai, walaupun pada matriks 3×3 shape-nya tidak berubah. Karena itu, shape saja tidak cukup untuk membuktikan operasi benar.

In [13]:
# Reproduksi versi NumPy jaringan berlapis, hlm. 43.
import numpy as np

ih_wgt = np.array([
    [0.1, 0.2, -0.1],
    [-0.1, 0.1, 0.9],
    [0.1, 0.4, 0.1],
]).T
hp_wgt = np.array([
    [0.3, 1.1, -0.3],
    [0.1, 0.2, 0.0],
    [0.0, 1.3, 0.1],
]).T
weights = [ih_wgt, hp_wgt]

def neural_network(input_data, weights):
    hid = input_data.dot(weights[0])
    pred = hid.dot(weights[1])
    return pred

toes = np.array([8.5, 9.5, 9.9, 9.0])
wlrec = np.array([0.65, 0.8, 0.8, 0.9])
nfans = np.array([1.2, 1.3, 0.5, 1.0])
input_data = np.array([toes[0], wlrec[0], nfans[0]])
hid = input_data.dot(weights[0])
pred = neural_network(input_data, weights)
print('Hidden layer:', hid)
print('Prediction:', pred)
print('Shapes:', input_data.shape, ih_wgt.shape, hid.shape, hp_wgt.shape, pred.shape)

stack_hidden_numpy = hid.copy()
stack_pred_numpy = pred.copy()
stack_weights_numpy = [ih_wgt.copy(), hp_wgt.copy()]

Hidden layer: [0.86  0.295 1.23 ]
Prediction: [0.2135 0.145  0.5065]
Shapes: (3,) (3, 3) (3,) (3, 3) (3,)


## 16. Pendalaman: dua lapisan linear tetap dapat digabung

Contoh jaringan berlapis ini belum memakai aktivasi nonlinear. Dengan orientasi NumPy:

$$
\hat{y}=(xW_1)W_2=x(W_1W_2)
$$

Sifat asosiatif perkalian matriks memungkinkan dua transformasi linear tersebut digabung menjadi satu matriks efektif. Jadi, menumpuk operasi linear saja belum memberikan kemampuan nonlinear baru.

**Kode tambahan berikut** memeriksa hubungan tersebut dengan bobot reproduksi. Pembahasan nonlinearitas diperdalam pada bab berikutnya; notebook ini tidak mengubah arsitektur asli dengan menambahkan aktivasi.

In [14]:
# Tambahan: kesetaraan komposisi linear.
x = np.array(stack_input)
W1, W2 = stack_weights_numpy
effective_weights = W1.dot(W2)
combined_pred = x.dot(effective_weights)
print('Two-step prediction:', stack_pred_numpy)
print('Combined-matrix prediction:', combined_pred)
print('Equivalent:', np.allclose(stack_pred_numpy, combined_pred))

Two-step prediction: [0.2135 0.145  0.5065]
Combined-matrix prediction: [0.2135 0.145  0.5065]
Equivalent: True


## 17. A quick primer on NumPy - membuat array

**Asal kode:** reproduksi hlm. 44. Array `a` dan `b` berbentuk `(4,)`, sedangkan `c` adalah matriks `(2, 4)`. Satu array 1D berbeda dari matriks satu baris `(1, 4)`.

`np.zeros((2, 4))` membentuk matriks nol, dan `np.random.rand(2, 5)` membentuk matriks angka acak. **Adaptasi:** `np.random.seed(42)` ditambahkan agar demonstrasi acak dapat diulang. Buku tidak menentukan seed, sehingga angka acak di sini tidak dimaksudkan sama persis dengan angka pada gambar buku.

**Rujukan:** Trask (2019), hlm. 44; [dokumentasi `numpy.random.seed`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html).

In [15]:
# Reproduksi array NumPy, hlm. 44; seed ditambahkan untuk reproduktibilitas.
import numpy as np

np.random.seed(42)
a = np.array([0, 1, 2, 3])
b = np.array([4, 5, 6, 7])
c = np.array([[0, 1, 2, 3], [4, 5, 6, 7]])
d = np.zeros((2, 4))
e = np.random.rand(2, 5)

print('a:', a)
print('b:', b)
print('c:\n', c)
print('d:\n', d)
print('e:\n', e)
print('Shapes:', a.shape, b.shape, c.shape, d.shape, e.shape)

a: [0 1 2 3]
b: [4 5 6 7]
c:
 [[0 1 2 3]
 [4 5 6 7]]
d:
 [[0. 0. 0. 0.]
 [0. 0. 0. 0.]]
e:
 [[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864]
 [0.15599452 0.05808361 0.86617615 0.60111501 0.70807258]]
Shapes: (4,) (4,) (2, 4) (2, 4) (2, 5)


## 18. Operasi elementwise dan broadcasting

**Asal kode:** reproduksi operasi hlm. 44-45. Operator `*` pada `ndarray` melakukan perkalian elementwise, bukan perkalian matriks. Jika ukuran sesuai untuk broadcasting, operasi tersebut dapat diterapkan pada array dengan shape berbeda.

Pada `a * c`, `a` memiliki shape `(4,)` dan `c` memiliki shape `(2, 4)`, sehingga vektor dapat diterapkan pada setiap baris. Pada `a * e`, panjang 4 tidak cocok dengan dimensi terakhir 5.

Contoh error yang disengaja ditangani menggunakan `try/except ValueError`. Jika operasi ternyata tidak menimbulkan error yang diharapkan, notebook menghentikan eksekusi agar masalah tidak tersembunyi.

In [16]:
# Reproduksi operasi hlm. 44-45; error yang diharapkan ditangani.
print('a * 0.1:', a * 0.1)
print('c * 0.2:\n', c * 0.2)
print('a * b:', a * b)
print('a * b * 0.2:', a * b * 0.2)
print('a * c:\n', a * c)

try:
    print(a * e)
except ValueError as error:
    broadcasting_error_observed = True
    print('Expected ValueError:', str(error))
else:
    raise AssertionError('Shape (4,) dan (2, 5) seharusnya tidak broadcast-compatible.')

a * 0.1: [0.  0.1 0.2 0.3]
c * 0.2:
 [[0.  0.2 0.4 0.6]
 [0.8 1.  1.2 1.4]]
a * b: [ 0  5 12 21]
a * b * 0.2: [0.  1.  2.4 4.2]
a * c:
 [[ 0  1  4  9]
 [ 0  5 12 21]]
Expected ValueError: operands could not be broadcast together with shapes (4,) (2,5) 


### Klarifikasi aturan broadcasting

Aturan umum NumPy membandingkan dimensi dari **kanan ke kiri**. Dua dimensi kompatibel jika sama besar atau salah satunya berukuran 1; dimensi yang tidak ada diperlakukan sebagai ukuran 1. Aturan ini lebih lengkap daripada penjelasan sederhana tentang jumlah kolom pada buku.

**Contoh tambahan:** shape `(2, 1)` dan `(1, 3)` dapat menghasilkan `(2, 3)` melalui broadcasting. Ini merupakan operasi elementwise, dengan aturan berbeda dari perkalian matriks.

**Rujukan:** [Dokumentasi resmi NumPy - Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html).

In [17]:
# Tambahan: dimensi berukuran 1 dapat diperluas melalui broadcasting.
left = np.array([[1], [2]])       # shape (2, 1)
right = np.array([[10, 20, 30]])  # shape (1, 3)
broadcast_result = left * right
print('Shapes:', left.shape, right.shape, '->', broadcast_result.shape)
print(broadcast_result)

# Tambahan: transpose array 1D tidak mengubahnya menjadi vektor kolom.
vector_1d = np.array([1, 2, 3])
print('1D:', vector_1d.shape, 'transpose:', vector_1d.T.shape)
print('Explicit column vector:', vector_1d.reshape(-1, 1).shape)

Shapes: (2, 1) (1, 3) -> (2, 3)
[[10 20 30]
 [20 40 60]]
1D: (3,) transpose: (3,)
Explicit column vector: (3, 1)


## 19. Perkalian matriks dan shape

Untuk matriks 2D, perkalian mengikuti:

$$
(m,d)\,(d,k)\longrightarrow(m,k)
$$

Dimensi dalam harus cocok. Dimensi luar menjadi shape hasil. `dot` untuk dua matriks 2D melakukan perkalian matriks, sedangkan `*` tetap elementwise.

**Asal kode:** contoh `(1,4).dot(4,3)` pada hlm. 45 dan contoh lanjutan hlm. 46. Variabel nol dari buku dipertahankan karena tujuan contohnya memeriksa bentuk, bukan nilai prediksi.

**Rujukan:** Trask (2019), hlm. 45-46; [dokumentasi NumPy `dot`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html).

In [18]:
# Reproduksi contoh shape pada hlm. 45.
a = np.zeros((1, 4))
b = np.zeros((4, 3))
c = a.dot(b)
print(c.shape)
shape_result_1 = c.shape

# Reproduksi contoh shape pada hlm. 46.
a = np.zeros((2, 4))
b = np.zeros((4, 3))
c = a.dot(b)
print(c.shape)

e = np.zeros((2, 1))
f = np.zeros((1, 3))
g = e.dot(f)
print(g.shape)

h = np.zeros((5, 4)).T
i = np.zeros((5, 6))
j = h.dot(i)
print(j.shape)

matrix_shape_results = [shape_result_1, c.shape, g.shape, j.shape]

(1, 3)
(2, 3)
(2, 3)
(4, 6)


In [19]:
# Reproduksi contoh shape yang salah, hlm. 46, dengan penanganan error.
h = np.zeros((5, 4))
i = np.zeros((5, 6))

try:
    j = h.dot(i)
    print(j.shape)
except ValueError as error:
    matrix_error_observed = True
    print('Expected ValueError:', str(error))
else:
    raise AssertionError('Dimensi dalam 4 dan 5 seharusnya tidak cocok.')

Expected ValueError: shapes (5,4) and (5,6) not aligned: 4 (dim 1) != 5 (dim 0)


### Perbedaan operasi yang perlu diingat

| Operasi | Contoh | Makna |
| --- | --- | --- |
| Scalar multiplication | `a * 0.1` | Setiap elemen dikalikan satu skalar |
| Elementwise multiplication | `a * b` | Perkalian per posisi setelah aturan broadcasting |
| Vector dot product | `x.dot(w)` untuk dua array 1D | Perkalian per posisi kemudian dijumlahkan |
| Matrix multiplication | `A.dot(B)` untuk dua array 2D | Menggabungkan baris dan kolom dengan weighted sum |
| Transpose | `W.T` untuk matriks 2D | Menukar sumbu baris dan kolom |

Aturan shape untuk dot product tidak boleh disamakan dengan aturan broadcasting untuk operator elementwise. Keberhasilan suatu operasi dijalankan juga belum membuktikan orientasi feature dan output telah sesuai dengan tujuan model.

**Rujukan:** Trask (2019), hlm. 44-46. Tabel merupakan sintesis penjelasan.

## 20. Pendalaman: satu sample dan beberapa sample

Kode utama buku mengambil pertandingan pertama. **Tambahan berikut** menggabungkan empat observasi yang sudah ada pada contoh menjadi matriks `(4, 3)` dan menerapkan bobot yang sama pada masing-masing baris. Ini memperjelas perbedaan banyak feature dengan banyak sample, bukan menambahkan proses training.

Pada sebagian observasi, hasil lebih besar dari 1. Hal ini memperlihatkan secara langsung bahwa weighted sum linear belum menjamin keluaran berupa probabilitas, walaupun ilustrasi buku memakai konteks peluang kemenangan.

In [20]:
# Tambahan: empat sample dengan tiga feature yang sama seperti contoh buku.
X = np.array([
    [8.5, 0.65, 1.2],
    [9.5, 0.8, 1.3],
    [9.9, 0.8, 0.5],
    [9.0, 0.9, 1.0],
])
batch_weights = np.array([0.1, 0.2, 0.0])
batch_predictions = X.dot(batch_weights)
print('Input shape:', X.shape)
print('Predictions:', batch_predictions)
print('Any output greater than 1:', bool(np.any(batch_predictions > 1)))

Input shape: (4, 3)
Predictions: [0.98 1.11 1.15 1.08]
Any output greater than 1: True


## 21. Catatan reproduksi dan penyesuaian

| Bagian | Penyesuaian dan alasan |
| --- | --- |
| Variabel `input` | Diganti `input_data` agar fungsi bawaan Python tidak tertimpa |
| Hlm. 37 dan 39 | Pemanggilan `weight` diperbaiki menjadi `weights`, sesuai bobot vektor/matriks |
| Listing yang terpotong pada diagram | Digabung menjadi sel lengkap dengan definisi sebelum pemanggilan |
| `ele_mul` dan `vect_mat_mul` buku | Batas contoh tiga output/persegi dipertahankan dan dijelaskan |
| Helper matriks umum | Dibuat terpisah serta diberi label tambahan |
| Tantangan vector math | Solusi pendamping diberikan; bukan listing solusi dari buku |
| Array acak hlm. 44 | Seed 42 ditambahkan; nilai acak tidak diklaim sama dengan gambar buku |
| Contoh error hlm. 44 dan 46 | Error yang diharapkan ditangani supaya Run all dapat selesai |
| Angka output | Output eksekusi asli disimpan; perbedaan floating point dijelaskan |

Data contoh sederhana berasal dari buku. Tidak ada file dataset eksternal atau bobot hasil pelatihan. Seluruh reproduksi hanya menghitung forward propagation menggunakan bobot yang telah ditetapkan.

**Rujukan:** Trask (2019), hlm. 24-46.

## 22. Verifikasi hasil reproduksi

Sel berikut membandingkan nilai penting dengan hasil matematis yang diharapkan serta memeriksa kesetaraan implementasi list dan NumPy. Toleransi numerik digunakan karena bilangan desimal direpresentasikan sebagai floating point.

Pemeriksaan ini membuktikan konsistensi **perhitungan dan implementasi contoh**, bukan akurasi model pada pertandingan nyata. Contoh error yang disengaja juga diperiksa telah teramati.

In [21]:
# Pemeriksaan numerik dan kesetaraan implementasi.
np.testing.assert_allclose(single_pred, 0.85, rtol=0, atol=1e-12)
np.testing.assert_allclose(multi_input_pred_python, 0.98, rtol=0, atol=1e-12)
np.testing.assert_allclose(multi_input_pred_python, multi_input_pred_numpy, rtol=0, atol=1e-12)
np.testing.assert_allclose(multi_output_pred_python, [0.195, 0.13, 0.585], rtol=0, atol=1e-12)
np.testing.assert_allclose(multi_io_pred_python, [0.555, 0.98, 0.965], rtol=0, atol=1e-12)

# NumPy menjadi pembanding independen untuk orientasi baris pada helper list.
multi_io_reference = np.array(multi_io_input).dot(np.array(multi_io_weights_rows).T)
np.testing.assert_allclose(multi_io_pred_python, multi_io_reference, rtol=0, atol=1e-12)
np.testing.assert_allclose(rectangular_pred, rectangular_numpy, rtol=0, atol=1e-12)

np.testing.assert_allclose(stack_hidden_python, [0.86, 0.295, 1.23], rtol=0, atol=1e-12)
np.testing.assert_allclose(stack_pred_python, [0.2135, 0.145, 0.5065], rtol=0, atol=1e-12)
np.testing.assert_allclose(stack_hidden_python, stack_hidden_numpy, rtol=0, atol=1e-12)
np.testing.assert_allclose(stack_pred_python, stack_pred_numpy, rtol=0, atol=1e-12)
np.testing.assert_allclose(combined_pred, stack_pred_numpy, rtol=0, atol=1e-12)

np.testing.assert_allclose(dot_from_vector_operations(vec_a, vec_b), np.dot(vec_a, vec_b))
assert dot_examples['c dot e'] == 0 and dot_examples['e dot e'] == 2
assert matrix_shape_results == [(1, 3), (2, 3), (2, 3), (4, 6)]
np.testing.assert_allclose(broadcast_result, [[10, 20, 30], [20, 40, 60]])
np.testing.assert_allclose(batch_predictions, [0.98, 1.11, 1.15, 1.08], rtol=0, atol=1e-12)
assert broadcasting_error_observed and matrix_error_observed
print('Semua pemeriksaan numerik dan perbandingan implementasi berhasil.')
print('Kedua contoh ValueError yang disengaja berhasil ditangani.')

Semua pemeriksaan numerik dan perbandingan implementasi berhasil.
Kedua contoh ValueError yang disengaja berhasil ditangani.


### Ringkasan nilai teoritis

| Contoh | Hasil |
| --- | --- |
| Satu input, satu output | 0.85 |
| Tiga input, satu output | 0.98 |
| Satu input, tiga output | `[0.195, 0.13, 0.585]` |
| Tiga input, tiga output | `[0.555, 0.98, 0.965]` |
| Hidden layer | `[0.86, 0.295, 1.23]` |
| Output jaringan berlapis | `[0.2135, 0.145, 0.5065]` |

**Status eksekusi tersimpan:** 21 sel kode dijalankan berurutan pada Python 3.12.14 dengan NumPy 2.3.5, tanpa exception yang tidak ditangani. Output tersimpan berasal dari eksekusi tersebut. Jalankan ulang di Colab untuk memastikan hasil pada lingkunganmu.

## 23. Summary

Forward propagation mengubah input menjadi prediksi melalui operasi bobot. Pada contoh satu input, operasi dasarnya adalah perkalian. Pada banyak input, kontribusi digabung melalui weighted sum. Banyak output memerlukan bobot untuk masing-masing keluaran, dan beberapa lapisan menggunakan keluaran antara sebagai masukan berikutnya.

Vektor dan matriks memberi cara untuk mengatur operasi tersebut. Implementasi NumPy mereproduksi perhitungan Python list jika orientasi dan shape konsisten. Operator elementwise, dot product, dan perkalian matriks memiliki aturan yang berbeda dan perlu dibaca sesuai bentuk datanya.

Seluruh bobot pada bab ini ditetapkan sebelumnya. Prediksi yang berhasil dihitung belum berarti model telah belajar atau memiliki akurasi yang baik.

**Rujukan:** Trask (2019), hlm. 46.